# 06. Fusion Pipeline


## What This Notebook Does

This notebook trains one simple model on top of the base models.

1. load the saved EEG, MEG, speech, and face models
2. collect their probability outputs
3. join those probabilities into one feature vector
4. train one Logistic Regression fusion model
5. compare it with very simple baselines
6. save the trained files


## Step 1: Setup


In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()
possible_dirs = [current_dir, current_dir / "NeuroSense" / "notebooks"]
notebooks_dir = next((path for path in possible_dirs if path.exists() and path.name == "notebooks"), None)
if notebooks_dir is None:
    raise FileNotFoundError("Start Jupyter from the project root or from NeuroSense/notebooks.")

if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook()
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print("Artifacts directory:", ARTIFACTS_DIR)


## Step 2: Import The Libraries


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from utils.emotion_utils import SENTIMENT_ORDER, aggregate_probabilities, map_emotion_to_sentiment


## Step 3: Check The Needed Files And Load The Base Models


In [ ]:
required_files = [
    ARTIFACTS_DIR / "eeg" / "eeg_model.pkl",
    ARTIFACTS_DIR / "eeg" / "eeg_scaler.pkl",
    ARTIFACTS_DIR / "eeg" / "eeg_label_encoder.pkl",
    ARTIFACTS_DIR / "speech" / "speech_model.pkl",
    ARTIFACTS_DIR / "speech" / "speech_scaler.pkl",
    ARTIFACTS_DIR / "speech" / "speech_label_encoder.pkl",
    ARTIFACTS_DIR / "face" / "face_model.pkl",
    ARTIFACTS_DIR / "face" / "face_scaler.pkl",
    ARTIFACTS_DIR / "face" / "face_pca.pkl",
    ARTIFACTS_DIR / "face" / "face_label_encoder.pkl",
    CACHE_DIR / "speech_features_deduped.npz",
    CACHE_DIR / "face_test_sentiment.npz",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    message = "Run notebooks 01 to 05 first. Missing files:\n" + "\n".join(missing_files)
    raise FileNotFoundError(message)

eeg_model = joblib.load(ARTIFACTS_DIR / "eeg" / "eeg_model.pkl")
eeg_scaler = joblib.load(ARTIFACTS_DIR / "eeg" / "eeg_scaler.pkl")
eeg_encoder = joblib.load(ARTIFACTS_DIR / "eeg" / "eeg_label_encoder.pkl")


speech_model = joblib.load(ARTIFACTS_DIR / "speech" / "speech_model.pkl")
speech_scaler = joblib.load(ARTIFACTS_DIR / "speech" / "speech_scaler.pkl")
speech_encoder = joblib.load(ARTIFACTS_DIR / "speech" / "speech_label_encoder.pkl")

face_model = joblib.load(ARTIFACTS_DIR / "face" / "face_model.pkl")
face_scaler = joblib.load(ARTIFACTS_DIR / "face" / "face_scaler.pkl")
face_pca = joblib.load(ARTIFACTS_DIR / "face" / "face_pca.pkl")
face_encoder = joblib.load(ARTIFACTS_DIR / "face" / "face_label_encoder.pkl")

print("All base models loaded.")


## Step 4: Build 3-Class Sentiment Probability Vectors

Every base model may use different labels, so we convert all probability outputs into the same three sentiments: `NEGATIVE`, `NEUTRAL`, and `POSITIVE`.


In [ ]:
def group_rows_by_sentiment(true_labels, probability_matrix, class_labels):
    grouped = {sentiment: [] for sentiment in SENTIMENT_ORDER}

    for true_label, probability_row in zip(true_labels, probability_matrix):
        true_sentiment = map_emotion_to_sentiment(true_label)
        sentiment_vector = aggregate_probabilities(class_labels, probability_row, order=SENTIMENT_ORDER)
        grouped[true_sentiment].append(sentiment_vector)

    return grouped


eeg_df = pd.read_csv(DATASETS_DIR / "eeg" / "eeg" / "emotions.csv")
eeg_X = eeg_df.drop(columns=["label"]).select_dtypes(include="number").fillna(0.0).values
eeg_y = eeg_df["label"].astype(str).values
_, eeg_X_test, _, eeg_y_test = train_test_split(eeg_X, eeg_y, test_size=0.2, stratify=eeg_y, random_state=RANDOM_STATE)
eeg_probs = eeg_model.predict_proba(eeg_scaler.transform(eeg_X_test))


speech_cache = np.load(CACHE_DIR / "speech_features_deduped.npz", allow_pickle=True)
speech_X = speech_cache["X"]
speech_y = speech_cache["y"].astype(str)
_, speech_X_test, _, speech_y_test = train_test_split(speech_X, speech_y, test_size=0.2, stratify=speech_y, random_state=RANDOM_STATE)
speech_probs = speech_model.predict_proba(speech_scaler.transform(speech_X_test))

face_cache = np.load(CACHE_DIR / "face_test_sentiment.npz", allow_pickle=True)
face_X_test = face_cache["X"]
face_y_test = face_cache["y"].astype(str)
face_probs = face_model.predict_proba(face_pca.transform(face_scaler.transform(face_X_test)))

modality_groups = {
    "eeg": group_rows_by_sentiment(eeg_y_test, eeg_probs, eeg_encoder.classes_),
    "speech": group_rows_by_sentiment(speech_y_test, speech_probs, speech_encoder.classes_),
    "face": group_rows_by_sentiment(face_y_test, face_probs, face_encoder.classes_),
}


## Step 5: Build The Fusion Dataset

One fusion row has 12 numbers: 3 probabilities from EEG, 3 from MEG, 3 from speech, and 3 from face.


In [ ]:
fusion_rows = []
fusion_labels = []

for sentiment in SENTIMENT_ORDER:
    eeg_rows = modality_groups["eeg"][sentiment]
    speech_rows = modality_groups["speech"][sentiment]
    face_rows = modality_groups["face"][sentiment]

    row_count = min(len(eeg_rows), len(speech_rows), len(face_rows))

    for row_index in range(row_count):
        fusion_row = np.concatenate([
            eeg_rows[row_index],
            speech_rows[row_index],
            face_rows[row_index],
        ])
        fusion_rows.append(fusion_row)
        fusion_labels.append(sentiment)

X_fusion = np.vstack(fusion_rows)
y_fusion = np.array(fusion_labels)

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_fusion)

X_train, X_test, y_train, y_test = train_test_split(
    X_fusion,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=RANDOM_STATE,
)

print("Fusion feature matrix shape:", X_fusion.shape)
print("Fusion class names:", list(encoder.classes_))


## Step 6: Train The Fusion Model


In [ ]:
model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred_meta = model.predict(X_test)
meta_accuracy = accuracy_score(y_test, y_pred_meta)


## Step 7: Compare With Simple Baselines

This helps us check whether the trained fusion model is really adding value.


In [ ]:
def average_baseline_predict(X_data, n_modalities=3, n_classes=3):
    predictions = []
    for row in X_data:
        row_matrix = row.reshape(n_modalities, n_classes)
        average_vector = row_matrix.mean(axis=0)
        predictions.append(int(np.argmax(average_vector)))
    return np.array(predictions)


def majority_vote_predict(X_data, n_modalities=3, n_classes=3):
    predictions = []
    for row in X_data:
        row_matrix = row.reshape(n_modalities, n_classes)
        votes = np.argmax(row_matrix, axis=1)
        vote_counts = np.bincount(votes, minlength=n_classes)
        predictions.append(int(np.argmax(vote_counts)))
    return np.array(predictions)


y_pred_average = average_baseline_predict(X_test)
y_pred_vote = majority_vote_predict(X_test)

average_accuracy = accuracy_score(y_test, y_pred_average)
vote_accuracy = accuracy_score(y_test, y_pred_vote)

print("Fusion model accuracy:", round(meta_accuracy, 4))
print("Average baseline accuracy:", round(average_accuracy, 4))
print("Majority vote accuracy:", round(vote_accuracy, 4))


## Step 8: Check The Result And Save The Files


In [ ]:
print(classification_report(y_test, y_pred_meta, target_names=encoder.classes_))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_meta,
    display_labels=encoder.classes_,
    cmap="YlGnBu",
    xticks_rotation=20,
)
plt.title("Fusion confusion matrix")
plt.tight_layout()
plt.show()

artifact_dir = ARTIFACTS_DIR / "fusion"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "fusion_model.pkl")
joblib.dump(encoder, artifact_dir / "fusion_label_encoder.pkl")

meta_model_adds_value = bool(meta_accuracy > average_accuracy)
metadata = {
    "data_source": "Late-fusion probability pooling",
    "data_source_note": "Each fusion row is built by joining probability outputs from EEG, MEG, speech, and face models.",
    "evaluation_method": "80/20 Stratified Random Split on fusion feature matrix",
    "meta_model_accuracy": round(float(meta_accuracy), 4),
    "average_baseline_accuracy": round(float(average_accuracy), 4),
    "majority_vote_accuracy": round(float(vote_accuracy), 4),
    "n_modalities": 3,
    "fusion_method": "late_probability_pooling",
    "meta_model_adds_value": meta_model_adds_value,
}
if not meta_model_adds_value:
    metadata["fallback_reason"] = "Meta-model did not outperform the simple average baseline."

with open(artifact_dir / "fusion_metadata.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Saved fusion files to:", artifact_dir)
